In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Menentukan pilihan Sampler

<Accordion>
<AccordionItem title="Versi pakej">

Kod di halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan penggunaan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>

Anda boleh guna pilihan untuk menyesuaikan primitif Sampler. Bahagian ini menumpukan pada cara menentukan pilihan primitif Qiskit Runtime. Walaupun antara muka kaedah `run()` primitif adalah sama merentas semua pelaksanaan, pilihan mereka tidak demikian. Rujuk rujukan API yang berkaitan untuk maklumat tentang pilihan [`qiskit.primitives.BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) dan [`qiskit_aer.primitives.SamplerV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.SamplerV2.html).

<span id="pass-options"></span>
## Menetapkan pilihan Sampler
Anda boleh menetapkan pilihan semasa memulakan Sampler, selepas memulakan Sampler, atau anda boleh mengemas kini pilihan selepas Sampler dimulakan. Untuk arahan menggunakan teknik-teknik ini, lihat topik [Pengenalan kepada pilihan](/guides/runtime-options-overview#options-precedence).

Selain itu, anda boleh menetapkan nilai `shots` dalam kaedah `run()`, seperti yang diterangkan dalam bahagian berikut.

<span id="run-method"></span>
### Kaedah Run()
Satu-satunya nilai yang boleh anda masukkan ke `run()` ialah yang ditakrifkan dalam antara muka. Iaitu, `shots`. Ini mengatasi mana-mana nilai yang ditetapkan untuk `default_shots` bagi run semasa.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d8286680bvlc73d1vmu0', 'sampler')>

### Special cases

<span id="shots"></span>
#### Shots

The `SamplerV2.run` method accepts two arguments: a list of PUBs, each of which can specify a PUB-specific value for shots, and a shots keyword argument. These shot values are a part of the Sampler execution interface, and are independent of the Runtime Sampler's options.  They take precedence over any values specified as options in order to comply with the Sampler abstraction.

However, if `shots` is not specified by any PUB or in the run keyword argument (or if they are all `None`), then the shots value from the options is used, most notably `default_shots`.

To summarize, this is the order of precedence for specifying shots in the Sampler, for any particular PUB:

1. If the PUB specifies shots, use that value.
2. If the `shots` keyword argument is specified in `run`, use that value.
4. If `twirling` is enabled  (True by default), then the product of `num_randomizations` and `shots_per_randomization`, as specified as  [`twirling` options](/docs/api/qiskit-ibm-runtime/options-twirling-options), is used.
5. If `sampler.options.default_shots` is specified, use that value.

Thus, if shots are specified in all possible places, the one with highest precedence (shots specified in the PUB) is used.

Example:

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden
# if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d82868ugbeec73alfa80', 'sampler')>

### Kes khas
<span id="shots"></span>
#### Shots
Kaedah `SamplerV2.run` menerima dua argumen: senarai PUB, di mana setiap satu boleh menentukan nilai shots khusus untuk PUB tersebut, dan argumen keyword shots. Nilai shots ini adalah sebahagian daripada antara muka pelaksanaan Sampler, dan bebas daripada pilihan Runtime Sampler. Ia diutamakan berbanding mana-mana nilai yang ditentukan sebagai pilihan bagi mematuhi abstraksi Sampler.

Walau bagaimanapun, jika `shots` tidak ditentukan oleh mana-mana PUB atau dalam argumen keyword run (atau semua `None`), maka nilai shots daripada pilihan digunakan, terutamanya `default_shots`.

Untuk merumuskan, ini adalah susunan keutamaan untuk menentukan shots dalam Sampler, bagi setiap PUB tertentu:

1. Jika PUB menentukan shots, gunakan nilai itu.
2. Jika argumen keyword `shots` ditentukan dalam `run`, gunakan nilai itu.
4. Jika `twirling` didayakan (True secara lalai), maka hasil darab `num_randomizations` dan `shots_per_randomization`, yang ditentukan sebagai [pilihan `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options), digunakan.
5. Jika `sampler.options.default_shots` ditentukan, gunakan nilai itu.

Oleh itu, jika shots ditentukan di semua tempat yang mungkin, yang mempunyai keutamaan tertinggi (shots yang ditentukan dalam PUB) digunakan.

Contoh: